# 🧠 Deep Q-Networks with Keras

In this notebook I build a **Deep Q-Network (DQN)** from primitives and use it to solve `CartPole-v1`:
keep a pole balanced upright on a moving cart for as long as possible.

Every piece is hand-written — the Q-network, the replay buffer, the exploration policy, the Bellman
update, and the training loop. No RL library does the thinking for me.

## 📋 Overview

Every other notebook in this module trained a network to map a fixed input to a fixed target: an
image to a label, a noisy image to a clean one, a latent vector to a fake sample. Reinforcement
learning removes the dataset entirely. There's an **agent** interacting with an **environment**,
taking actions, and only finding out afterward — through a **reward** — whether that was a good idea.

No labelled "correct action" ever exists. The network has to estimate how good each action is from a
given state, and improve that estimate over thousands of attempts using data it generates itself.

This is close to control engineering: a controller observing a plant, applying an input, measuring
the response, adjusting. The difference is that I don't derive the control law analytically — the
network learns it from experience.

| What I build | Why it exists |
|---|---|
| 🌍 Gymnasium `CartPole-v1` environment | The plant I'm trying to control |
| 🏗️ Keras Q-network | Estimates the value of each action from a continuous state |
| 📥 Replay buffer | Decorrelates training samples so gradient descent behaves |
| 🎯 Epsilon-greedy policy | Balances trying new things vs. using what I know |
| 🔄 Bellman update | Turns reinforcement learning into a regression problem |
| ⚙️ Training loop | Where the agent generates its own training data |
| 📊 Evaluation loop | Measures the trained policy with exploration switched off |

**What CartPole actually is.** A cart slides along a frictionless track with a pole hinged on top.
The state is 4 continuous numbers:

$$s = [x, \dot{x}, \theta, \dot{\theta}]$$

Where:
- $x$ = cart position
- $\dot{x}$ = cart velocity
- $\theta$ = pole angle from vertical
- $\dot{\theta}$ = pole angular velocity

Two actions: push left (0), push right (1). Every timestep the pole stays up earns $+1$ reward. The
episode ends when $|\theta| > 0.20948$ rad ($\approx 12°$) or $|x| > 2.4$.

Those two thresholds come back later — I use them to build a continuous shaped reward in Practice 3.

## 🧩 Theory

### The reinforcement learning loop

At each timestep $t$ the agent observes state $s_t$, picks action $a_t$, and the environment returns
a reward $r_t$ and a next state $s_{t+1}$:

```
        action a_t
  ┌──────────────────────►┌─────────────┐
  │                       │ ENVIRONMENT │
┌─┴─────┐                 │  (CartPole) │
│ AGENT │                 └──────┬──────┘
└─▲─────┘                        │
  │   state s_{t+1}, reward r_t  │
  └──────────────────────────────┘
```

A closed feedback loop — structurally the same shape as an automatic gain control or a power-control
loop in a radio link. The agent is the controller, the reward is the error signal, and the policy is
the control law being tuned online.

### 🔢 Q-values and the Bellman equation

The **Q-value** $Q(s,a)$ answers one question: *if I'm in state $s$ and take action $a$, how much
total reward should I expect from here to the end of the episode?*

"Total" needs discounting, or the sum can diverge and reward 200 steps away counts the same as
reward right now:

$$Q(s_t, a_t) = \mathbb{E}\left[\sum_{k=0}^{\infty} \gamma^k \, r_{t+k}\right]$$

$\gamma \in [0,1)$ is the **discount factor**. With $\gamma = 0.95$, reward 20 steps out is worth
$0.95^{20} \approx 0.36$ of the same reward now.

This is exactly an exponential forgetting factor — the same structure as the decay in an
exponentially-weighted moving average used to smooth a noisy RSSI measurement. A useful rule of
thumb is the **effective horizon** $H \approx 1/(1-\gamma)$, so $\gamma = 0.95$ means the agent
meaningfully plans about 20 steps ahead. On CartPole, where failure is always roughly that far away,
that's a well-matched choice rather than an arbitrary one.

The **Bellman equation** is the recursion that makes this computable — the value of acting now is
the immediate reward plus the discounted value of acting optimally from wherever I land:

$$Q(s, a) = r + \gamma \max_{a'} Q(s', a')$$

The classic **tabular** Q-learning update nudges a stored table entry toward that target:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\Big[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,\Big]$$

### 🏗️ Why a neural network instead of a table

That table works when states are discrete and countable. CartPole's state is **4 continuous real
numbers** — there is no finite table to fill in, and the agent will essentially never visit the exact
same state twice.

So I replace the table with a function approximator:

$$Q(s,a) \;\approx\; Q_\theta(s,a)$$

A network with weights $\theta$ that takes a state vector and outputs one Q-value per action. It
**generalizes**: a state it has never seen still produces a sensible estimate, because nearby states
produce nearby outputs. That's the whole reason "Deep" appears in Deep Q-Network.

### 📉 Turning RL into regression

Here's the trick that makes this trainable with ordinary Keras. Treat the right-hand side of the
Bellman equation as a **target label** and fit the network to it with mean squared error:

$$y = \begin{cases}
r & \text{if the episode ended} \\[4pt]
r + \gamma \max_{a'} Q_\theta(s', a') & \text{otherwise}
\end{cases}$$

$$\mathcal{L}(\theta) = \big(Q_\theta(s,a) - y\big)^2$$

Now it's supervised regression — except the labels come from the network's own predictions and move
as training progresses. That self-reference is the main source of DQN instability, and it's worth
remembering when training looks erratic.

The terminal branch matters: a terminal state has no future, so bootstrapping from it would invent
value that doesn't exist.

### 🎯 Epsilon-greedy exploration

If the agent always takes the action it currently rates highest, it locks onto whatever it believed
early and never discovers better options. So with probability $\epsilon$ it acts randomly:

$$a = \begin{cases}
\text{random action} & \text{with probability } \epsilon \\[4pt]
\arg\max_a Q_\theta(s,a) & \text{with probability } 1 - \epsilon
\end{cases}$$

$\epsilon$ starts at $1.0$ and decays multiplicatively toward a floor:

$$\epsilon \leftarrow \max(\epsilon_{\min},\; \epsilon \cdot \lambda), \qquad \lambda = 0.995$$

The classic explore/exploit tradeoff — the same tension as a spectrum scan sweeping for a better
channel versus staying on the one that currently works. Sweep too much and throughput suffers; never
sweep and you sit on a degraded channel forever.

### 📥 Experience replay

Training on each transition as it happens breaks gradient descent: consecutive timesteps are almost
identical and heavily correlated, and SGD assumes roughly i.i.d. samples.

The fix is a **replay buffer** — store every transition $(s, a, r, s', \text{done})$ in a fixed-size
FIFO queue, then train on *random minibatches* sampled from it. This decorrelates consecutive samples
and lets each experience be reused many times.

Think of it as an interleaver in a communication system: burst errors get spread across the stream so
the decoder sees something closer to independent noise instead of one catastrophic clump.

### ⚠️ The piece I'm deliberately leaving out: the target network

A production DQN uses a **second, frozen copy** of the network to compute $\max_{a'}Q(s',a')$,
re-synced every $N$ steps. Without it, the same weights produce both the prediction and the label —
the network chases a target that moves every time it updates. It's the difference between calibrating
against a stable reference oscillator and calibrating against another instrument that's being
calibrated at the same time.

I'm keeping the single-network version here because it makes the core mechanism visible. The gap is
real, it's named in the Summary, and closing it is the first Sandbox exercise.

| Concept | Symbol | Value used here |
|---|---|---|
| Discount factor | $\gamma$ | 0.95 (effective horizon ≈ 20 steps) |
| Exploration rate | $\epsilon$ | 1.0 → 0.01 |
| Decay rate | $\lambda$ | 0.995 |
| Replay capacity | — | 2000 transitions |
| Minibatch size | — | 64 |
| Learning rate | $\alpha$ | 0.001 (Adam) |

## Part 1 — 🌍 Setting Up the Environment

[Gymnasium](https://gymnasium.farama.org/) is the maintained successor to OpenAI Gym. It gives every
environment the same `reset()` / `step(action)` interface, so agent code is portable across problems.

I pin versions first — Gymnasium, TensorFlow and NumPy have had compatibility friction in hosted
notebook environments, so this fixes a known-working combination before anything is imported.

In [ ]:
# Install/pin compatible versions — Gymnasium + TensorFlow + NumPy have had
# version-compatibility friction in hosted notebook environments, so this
# pins a known-working combination before importing anything.
!pip install --upgrade numpy==1.26.4 --quiet
!pip install gymnasium --quiet

In [ ]:
import numpy as np
import random
from collections import deque

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

import matplotlib.pyplot as plt
import gymnasium as gym

# Reproducibility — RL has two independent randomness sources (the agent's own
# exploration and the environment's initial conditions), so both need seeding.
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

In [ ]:
env = gym.make('CartPole-v1')

# Cast explicitly to Python int -- Gymnasium's Discrete.n (and some
# observation_space.shape entries) can come back as numpy.int64, which
# Keras 3's Dense layer rejects for `units` even though the value itself
# (e.g. 2) is perfectly valid.
state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)

print(f"State size: {state_size}")
print(f"Action size: {action_size}")

📝 **On the `int()` casts.** This is a real compatibility trap, not defensive noise. Gymnasium returns
`numpy.int64` for `action_space.n`, and Keras 3 rejects that type for a layer's `units` argument even
though the *value* is fine. The error message points at the Dense layer, not at Gymnasium, which makes
it annoying to track down.

Seeding all three RNGs matters more than usual here. In supervised learning the data is fixed; in RL
the agent's own randomness *shapes what data gets collected*, so two unseeded runs can diverge
completely.

## Part 2 — 🏗️ Defining the Q-Network

The network maps a 4-dimensional state to 2 Q-values, one per action:

$$\mathbb{R}^4 \;\longrightarrow\; \text{Dense}(24, \text{relu}) \;\longrightarrow\; \text{Dense}(24, \text{relu}) \;\longrightarrow\; \mathbb{R}^2$$

Two design choices differ from a typical classifier and both matter:

- **Linear output activation, not softmax.** Q-values are unbounded real numbers estimating expected
  return — not a probability distribution, and they must not be squashed into one.
- **MSE loss, not cross-entropy.** This is regression onto a continuous target.

In [ ]:
def build_model(state_size, action_size):
    model = Sequential([
        Input(shape=(state_size,)),
        Dense(24, activation='relu'),
        Dense(24, activation='relu'),
        Dense(action_size, activation='linear')
    ])
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

model = build_model(state_size, action_size)
model.summary()

📝 **Layer by layer:**

| Element | Choice | Reason |
|---|---|---|
| `Input(shape=(state_size,))` | Explicit input layer | Modern Keras idiom; clearer than passing `input_dim` to the first `Dense` |
| `Dense(24, relu)` ×2 | Fully connected hidden layers | 4 inputs is a tiny state space — 24 units is plenty |
| `Dense(action_size, linear)` | 2 outputs | One Q-value per action, unbounded |
| `loss='mse'` | Mean squared error | Regression onto the Bellman target |
| `Adam(lr=0.001)` | Adaptive optimizer | Per-parameter learning rates, robust default |

The network is deliberately small. With a 4-number state and 2 actions this is a low-dimensional
regression problem — the difficulty lives in the *training signal*, not in model capacity. Practice 1
tests that claim directly by widening it to 32 units.

## Part 3 — 📥 Replay Buffer and Epsilon-Greedy Policy

Three pieces work together:

| Piece | Role |
|---|---|
| `memory` (deque) | Fixed-size FIFO buffer of past transitions |
| `remember()` | Appends `(state, action, reward, next_state, done)` |
| `act()` | Epsilon-greedy selection — random w.p. $\epsilon$, else $\arg\max_a Q(s,a)$ |

A `deque` with `maxlen=2000` gives a ring buffer for free: once full, appending drops the oldest
transition automatically. No manual index bookkeeping.

In [ ]:
gamma = 0.95         # discount factor
epsilon = 1.0        # initial exploration rate
epsilon_min = 0.01   # exploration floor
epsilon_decay = 0.995
memory = deque(maxlen=2000)

def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def act(state):
    global epsilon
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model.predict(state, verbose=0)
    return np.argmax(q_values[0])

📝 **Why `done` has to be stored.** The Bellman target branches on it — a terminal state has no future
to bootstrap from. Forgetting this flag is a classic DQN bug: the agent starts assigning future value
to states that by definition have none.

📝 **On the epsilon floor.** `epsilon_min = 0.01` means exploration never fully stops. That's
deliberate — a permanently frozen policy can't notice if the environment shifts.

## Part 4 — 🔄 The Q-Learning Update

For each transition in a random minibatch:

1. Compute the Bellman target $y = r + \gamma \max_{a'} Q_\theta(s', a')$, or $y = r$ if terminal.
2. Get the network's current predictions $Q_\theta(s, \cdot)$ for all actions.
3. **Overwrite only the entry for the action actually taken** with $y$.
4. Fit one step on the modified target vector.

Step 3 is the subtle one. I only have evidence about the action I took — I learned nothing about the
action I *didn't* take. Leaving that output equal to the network's own current prediction makes its
squared error exactly zero, so it contributes no gradient. The update is surgical: only the taken
action's Q-value moves.

**This implementation is vectorized.** The naive version loops over the minibatch calling
`predict()` and `fit()` once per transition — for a batch of 64 that's 192 separate Keras calls, each
carrying full framework overhead. Batching everything into 2 predicts and 1 fit does identical maths
far faster.

In [ ]:
def replay(batch_size):
    global epsilon
    if len(memory) < batch_size:
        return

    minibatch = random.sample(memory, batch_size)

    states = np.array([t[0][0] for t in minibatch])
    actions = np.array([t[1] for t in minibatch])
    rewards = np.array([t[2] for t in minibatch])
    next_states = np.array([t[3][0] for t in minibatch])
    dones = np.array([t[4] for t in minibatch])

    # Two batched predictions instead of 2 x batch_size individual ones.
    # q_values becomes the target vector: every entry keeps the network's own
    # prediction except the action actually taken, which is overwritten below.
    q_values = model.predict(states, verbose=0)
    q_values_next = model.predict(next_states, verbose=0)

    for i in range(batch_size):
        target = rewards[i]
        if not dones[i]:
            target += gamma * np.amax(q_values_next[i])
        q_values[i][actions[i]] = target

    model.fit(states, q_values, epochs=1, verbose=0)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

📝 **Line by line:**

| Line | What it does |
|---|---|
| `random.sample(memory, batch_size)` | Draws 64 transitions uniformly — this is what breaks temporal correlation |
| `t[0][0]` / `t[3][0]` | Strips the batch dimension: stored states have shape `(1, 4)`, stacking needs `(4,)` |
| `model.predict(states)` | One call for the whole batch instead of 64 |
| `target = rewards[i]` | Terminal case — no future to bootstrap from |
| `target += gamma * np.amax(...)` | Non-terminal case — the Bellman backup |
| `q_values[i][actions[i]] = target` | Surgical overwrite; untouched entries produce zero gradient |
| `model.fit(states, q_values)` | One gradient step on the full batch |
| `epsilon *= epsilon_decay` | Shift gradually from exploring to exploiting |

⚠️ **The known gap.** `q_values_next` comes from `model` — the same network being trained. A proper
DQN would use a frozen target network here. Sandbox exercise 1 closes this.

## Part 5 — ⚙️ Training the Agent

Each episode resets the environment, then steps through it up to `max_timesteps`, choosing actions
epsilon-greedily and remembering every transition.

Two things worth noticing in this loop:

**Reward shaping is already present.** The environment gives $+1$ per timestep, but the loop
overrides it to $-10$ whenever the episode ends early. The default signal carries no information
about *how close to failing* the agent is — a pole at 11.9° and one at 0.1° earn identical reward
right up until one ends the episode. An explicit failure penalty is a much sharper signal. Practice 3
replaces this binary version with a continuous one.

**Training happens periodically, not every step.** `train_frequency = 5` means `replay()` runs every
5th timestep rather than constantly, which cuts training cost substantially without much loss.

I've also added an **early stopping** check. The CartPole convention is 195+ steps sustained over 100
consecutive episodes. Requiring a *streak* rather than a single good episode is the point — initial
conditions vary, so one lucky run proves nothing. It's a debouncing condition: act on a sustained
threshold crossing, not a single one.

In [ ]:
episodes = 10
max_timesteps = 200
batch_size = 64
train_frequency = 5

# Early stopping: a sustained streak, not a single good episode
consecutive_success_threshold = 100   # episodes required in a row
success_episode_length = 195          # steps for an episode to count as a success
episode_lengths = []

for e in range(episodes):
    state, _ = env.reset()
    state = np.reshape(state, [1, state_size])
    total_reward = 0
    episode_length = 0

    for time in range(max_timesteps):
        action = act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])

        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        episode_length = time + 1

        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

        if time % train_frequency == 0:
            replay(batch_size)

    # Recorded OUTSIDE the inner loop, so episodes that survive the full cap
    # without terminating still get counted.
    episode_lengths.append(episode_length)

    if len(episode_lengths) >= consecutive_success_threshold and all(
        length >= success_episode_length
        for length in episode_lengths[-consecutive_success_threshold:]
    ):
        print(f"Early stopping at episode {e+1}: agent consistently reaches max episode length. ✅")
        break

env.close()

📝 **What to watch:**

- **Score** should trend upward — noisily. RL learning curves are far rougher than supervised ones,
  because the training data distribution shifts as the policy changes.
- **Epsilon** should fall steadily from 1.0. Early episodes are essentially random flailing; that's
  expected and necessary.

📝 **On `terminated` vs `truncated`.** Gymnasium separates these deliberately and conflating them is a
genuine correctness bug. `terminated` means the pole actually fell. `truncated` means the time limit
was reached *while the pole was still up*. Treating a truncation as terminal tells the agent there was
no future value at exactly the moment things were going well. Here `done = terminated or truncated`
correctly ends the episode either way — but note that a fully correct Bellman branch would use
`terminated` alone, since truncation doesn't imply zero future value. Sandbox exercise 3.

⚠️ **`episodes = 10` is a demonstration budget, not a training budget.** Ten episodes is nowhere near
enough to solve CartPole — the early stopping check above will not fire. The mechanism is correct; the
compute isn't there. Raise to several hundred (with a target network) if the goal is actual performance.

## Part 6 — 📊 Evaluating the Trained Agent

Evaluation differs from training in one critical way: **exploration is switched off**. The agent acts
purely greedily via `argmax`. No epsilon, no random actions, no learning — this measures the policy as
it actually stands.

I use a separate environment instance and a higher timestep cap (500) so a genuinely good policy isn't
artificially truncated at 200.

⚠️ **On `render()`:** it requires the environment to have been created with a `render_mode`, e.g.
`gym.make('CartPole-v1', render_mode='human')`. Created without one, `.render()` will either no-op or
warn depending on the Gymnasium version. I've kept the call in place but it's worth setting
`render_mode` explicitly if visual playback is the goal.

In [ ]:
eval_env = gym.make('CartPole-v1')
eval_episodes = 10
eval_max_timesteps = 500
eval_scores = []

for e in range(eval_episodes):
    state, _ = eval_env.reset()
    state = np.reshape(state, [1, state_size])
    total_reward = 0

    for time in range(eval_max_timesteps):
        eval_env.render()
        q_values = model.predict(state, verbose=0)
        action = np.argmax(q_values[0])

        next_state, reward, terminated, truncated, _ = eval_env.step(action)
        done = terminated or truncated
        state = np.reshape(next_state, [1, state_size])
        total_reward += reward

        if done:
            print(f"Evaluation Episode: {e+1}/{eval_episodes}, Score: {time}")
            break

    # Appended outside the `if done` branch so full-length episodes are counted
    eval_scores.append(total_reward)

print(f"\nAverage: {np.mean(eval_scores):.2f}, Max: {np.max(eval_scores)}, Min: {np.min(eval_scores)}")
eval_env.close()

📝 **Reading the numbers:**

| Metric | What it tells me |
|---|---|
| **Average reward** | Overall policy quality across episodes |
| **Max reward** | Best case — what the agent achieves when conditions favour it |
| **Min reward** | Worst case — how badly it fails on an unlucky start |

A **large max–min gap** is the interesting signal: the policy is brittle, handling some initial
conditions well and collapsing on others. That's the RL equivalent of a controller stable in the
nominal operating region but not across the full envelope.

CartPole-v1 is conventionally "solved" at an average of 195+ over 100 consecutive episodes. With a
10-episode training budget I should expect to land far short — this notebook is about the mechanism,
not the score.

## 🎯 Practice 1 — Network Architecture: 24 vs 32 Units

**Question:** does a wider network (32 units per hidden layer instead of 24) learn a better policy?

My claim in Part 2 was that capacity isn't the bottleneck on a 4-dimensional state — the training
signal is. This tests that directly.

⚠️ **Correcting a broken source solution.** The original exercise solution redefined `act()` as
`return env.action_space.sample()` — always a fully random action, completely ignoring the model's
predictions. That silently destroys the experiment: with that `act()`, *neither* network's Q-values
are ever used to choose an action, so the comparison measures nothing. I've kept genuine
epsilon-greedy selection below.

In [ ]:
def build_model_wide(state_size, action_size):
    model = Sequential([
        Input(shape=(state_size,)),
        Dense(32, activation='relu'),
        Dense(32, activation='relu'),
        Dense(action_size, activation='linear')
    ])
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))
    return model

model_wide = build_model_wide(state_size, action_size)
model_wide.summary()

In [ ]:
# Fresh exploration/memory state so this comparison isn't contaminated
# by the epsilon decay and replay buffer from the earlier training run.
epsilon = 1.0
memory_wide = deque(maxlen=2000)

def remember_wide(state, action, reward, next_state, done):
    memory_wide.append((state, action, reward, next_state, done))

def act_wide(state):
    # Kept as genuine epsilon-greedy action selection — NOT the original exercise's
    # env.action_space.sample()-only stub, which would make this comparison
    # meaningless (see note above).
    global epsilon
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model_wide.predict(state, verbose=0)
    return np.argmax(q_values[0])

def replay_wide(batch_size):
    global epsilon
    if len(memory_wide) < batch_size:
        return

    minibatch = random.sample(memory_wide, batch_size)
    states = np.array([t[0][0] for t in minibatch])
    actions = np.array([t[1] for t in minibatch])
    rewards = np.array([t[2] for t in minibatch])
    next_states = np.array([t[3][0] for t in minibatch])
    dones = np.array([t[4] for t in minibatch])

    q_values = model_wide.predict(states, verbose=0)
    q_values_next = model_wide.predict(next_states, verbose=0)

    for i in range(batch_size):
        target = rewards[i]
        if not dones[i]:
            target += gamma * np.amax(q_values_next[i])
        q_values[i][actions[i]] = target

    model_wide.fit(states, q_values, epochs=1, verbose=0)
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

env_wide = gym.make('CartPole-v1')

for e in range(episodes):
    state, _ = env_wide.reset()
    state = np.reshape(state, [1, state_size])

    for time in range(max_timesteps):
        action = act_wide(state)
        next_state, reward, terminated, truncated, _ = env_wide.step(action)
        done = terminated or truncated
        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])

        remember_wide(state, action, reward, next_state, done)
        state = next_state

        if done:
            print(f"[Wide net] Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

        if time % train_frequency == 0:
            replay_wide(batch_size)

env_wide.close()

📝 **Interpreting this fairly.** Over 10 episodes the difference between 24 and 32 units will be
dominated by random variation, not architecture. Drawing a conclusion from a single short run of each
would be reading noise. A real comparison needs multiple seeds and far more episodes.

That caveat *is* the lesson: parameter count is rarely the limiting factor on a low-dimensional
problem like this. The replay buffer, exploration schedule, and reward design matter far more.

## ⚙️ Practice 2 — Epsilon Scheduling: Open-Loop vs Closed-Loop

There are two fundamentally different ways to decide how fast exploration should decay.

**Time-based (open-loop).** Epsilon follows a fixed schedule determined only by the episode number,
regardless of how the agent is doing:

$$\epsilon_{\text{linear}} \leftarrow \max(\epsilon - 0.01,\; 0.01)$$

$$\epsilon_{\text{exponential}} \leftarrow \max(\epsilon \cdot 0.99,\; 0.01)$$

A **hybrid** runs linear first for a decisive early cut, then switches to exponential for a gentle
approach to the floor.

**Performance-based (closed-loop).** Epsilon responds to how well the agent is actually doing — decay
faster when recent scores are high, at the normal rate when still struggling.

This is precisely the open-loop vs closed-loop distinction from power control in a cellular link.
Open-loop sets transmit power from a fixed path-loss estimate; closed-loop adjusts it from actual
feedback about received quality. Open-loop is simpler and predictable; closed-loop adapts to reality
but can misbehave if the feedback signal is noisy — and an RL episode score is a *very* noisy signal.

| Schedule | Driven by | Early behaviour | Late behaviour |
|---|---|---|---|
| **Linear** | Episode count | Steady, constant absolute step | Hits the floor abruptly |
| **Exponential** | Episode count | Fast while $\epsilon$ is large | Long slow tail |
| **Hybrid** | Episode count | Decisive early cut | Gentle final approach |
| **Adaptive** | Recent score | Depends entirely on performance | Can stall if the agent plateaus |

In [ ]:
def decay_epsilon(epsilon, episode, switch_episode=100):
    """Time-based (open-loop): linear decay, then exponential after switch_episode."""
    if episode < switch_episode:
        return max(epsilon - 0.01, 0.01)   # Linear decay
    else:
        return max(epsilon * 0.99, 0.01)   # Exponential decay


def adjust_epsilon(score, success_threshold=200):
    """Performance-based (closed-loop): decay faster when the agent is doing well."""
    global epsilon
    if score >= success_threshold:
        epsilon *= 0.9              # doing well -> explore less, faster
    else:
        epsilon *= epsilon_decay    # still learning -> normal rate
    epsilon = max(epsilon_min, epsilon)

First I compare the three time-based schedules in isolation — no training, just the arithmetic, so the
shapes are visible rather than inferred from numbers.

In [ ]:
eps_linear, eps_exponential, eps_hybrid = 1.0, 1.0, 1.0
history = {'linear': [], 'exponential': [], 'hybrid': []}

for episode in range(300):
    eps_linear = max(eps_linear - 0.01, 0.01)
    eps_exponential = max(eps_exponential * 0.99, 0.01)
    eps_hybrid = decay_epsilon(eps_hybrid, episode, switch_episode=100)

    history['linear'].append(eps_linear)
    history['exponential'].append(eps_exponential)
    history['hybrid'].append(eps_hybrid)

for ep in [0, 50, 99, 100, 150, 200, 299]:
    print(f"Episode {ep:>3} | linear={history['linear'][ep]:.4f} "
          f"| exponential={history['exponential'][ep]:.4f} "
          f"| hybrid={history['hybrid'][ep]:.4f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history['linear'], label='Linear decay', linewidth=2)
plt.plot(history['exponential'], label='Exponential decay', linewidth=2)
plt.plot(history['hybrid'], label='Hybrid (linear -> exponential @100)', linewidth=2, linestyle='--')
plt.axvline(100, color='gray', linestyle=':', label='Switch point')
plt.xlabel('Episode')
plt.ylabel('Epsilon (exploration rate)')
plt.title('Time-Based Epsilon Decay Schedules (open-loop)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Now the closed-loop version in an actual training run. Epsilon is adjusted from the episode score
rather than the episode number.

In [ ]:
epsilon = 1.0
memory_adaptive = deque(maxlen=2000)
env_adaptive = gym.make('CartPole-v1')
adaptive_epsilon_history = []

for e in range(episodes):
    state, _ = env_adaptive.reset()
    state = np.reshape(state, [1, state_size])

    for time in range(max_timesteps):
        if np.random.rand() <= epsilon:
            action = random.randrange(action_size)
        else:
            q_values = model.predict(state, verbose=0)
            action = np.argmax(q_values[0])

        next_state, reward, terminated, truncated, _ = env_adaptive.step(action)
        done = terminated or truncated

        reward = reward if not done else -10
        next_state = np.reshape(next_state, [1, state_size])
        memory_adaptive.append((state, action, reward, next_state, done))
        state = next_state

        if done:
            adjust_epsilon(time)   # closed-loop: driven by the score just achieved
            adaptive_epsilon_history.append(epsilon)
            print(f"[Adaptive epsilon] Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

env_adaptive.close()

📝 **The failure mode of closed-loop here.** `adjust_epsilon` only accelerates decay when
`score >= 200` — the maximum possible under `max_timesteps = 200`. Early in training that condition
essentially never fires, so the adaptive schedule collapses to the plain exponential one. Worse, if
the agent plateaus below threshold, epsilon keeps decaying at the base rate and the agent stops
exploring while still performing badly — the exact opposite of what's wanted.

That's a general lesson about feedback control on a noisy measurement: the loop is only as good as
the signal driving it. A more robust version would trigger on a *rolling mean* of recent scores rather
than a single episode, which is the same reason a power-control loop filters its quality estimate
before acting on it.

## 🧪 Practice 3 — Continuous Reward Shaping

The training loop above uses a sparse reward: $+1$ per timestep, $-10$ on failure. The agent learns
nothing about *how close to failing* it is until the moment it fails.

A **dense** reward built directly from the state gives graded feedback on every single step:

$$r = \Big(1 - \frac{|x|}{2.4}\Big) + \Big(1 - \frac{|\theta|}{0.20948}\Big)$$

Where $x$ is cart position and $\theta$ is pole angle. The constants $2.4$ and $0.20948$ rad
($\approx 12°$) are **exactly CartPole-v1's own termination thresholds**. So each term is $0$ right at
the edge of failure and climbs toward $1$ as the agent approaches dead-centre and perfectly upright.

That choice of constants is what makes this principled rather than arbitrary — the reward is a
normalized distance-from-failure on each axis. It's the same instinct as a proportional control term:
penalize the error continuously instead of waiting for the system to hit a hard limit.

In [ ]:
def custom_reward(state):
    x, x_dot, theta, theta_dot = state
    reward = (1 - abs(x) / 2.4) + (1 - abs(theta) / 0.20948)
    return reward

In [ ]:
epsilon = 1.0
memory_custom = deque(maxlen=2000)
env_custom = gym.make('CartPole-v1')

for e in range(episodes):
    state, _ = env_custom.reset()
    state = np.reshape(state, [1, state_size])

    for time in range(max_timesteps):
        if np.random.rand() <= epsilon:
            action = random.randrange(action_size)
        else:
            q_values = model.predict(state, verbose=0)
            action = np.argmax(q_values[0])

        # The environment's own reward is discarded (_) in favour of the shaped one
        next_state, _, terminated, truncated, _ = env_custom.step(action)
        done = terminated or truncated

        reward = custom_reward(next_state) if not done else -10
        next_state_reshaped = np.reshape(next_state, [1, state_size])

        memory_custom.append((state, action, reward, next_state_reshaped, done))
        state = next_state_reshaped

        if epsilon > epsilon_min:
            epsilon *= epsilon_decay

        if done:
            print(f"[Custom reward] Episode: {e+1}/{episodes}, Score: {time}, Epsilon: {epsilon:.3f}")
            break

env_custom.close()

📝 **Note the raw vs reshaped state.** `custom_reward()` takes the raw 4-value vector straight from
`env.step()`, *before* the `np.reshape(..., [1, state_size])`. Passing the reshaped `(1, 4)` array
would make the tuple unpacking `x, x_dot, theta, theta_dot = state` fail. Easy mistake, unhelpful
error message.

⚠️ **Reward shaping changes the objective — treat it with suspicion.** The agent optimizes whatever I
actually wrote, not what I meant. If the shaping term is misaligned with the real goal, the agent will
find a way to farm the shaped reward while ignoring the actual task. Here both terms are monotonic in
distance-from-failure, so they're aligned. That alignment is a design property I have to verify, not
something I get for free.

| Reward design | Signal density | Risk |
|---|---|---|
| `+1` per timestep (environment default) | Sparse, late | Slow learning — feedback only at failure |
| `+1` / `-10` on failure (Part 5) | Sparse, sharper terminal signal | Still no gradient during the episode |
| Continuous position + angle (here) | Dense, every step | Must stay aligned with the true goal |

## 📊 Summary

I built a Deep Q-Network from primitives and used it to learn a control policy for CartPole.

### Components

| Concept | What it does | Telecom / RF analogy 📡 |
|---|---|---|
| Q-value $Q(s,a)$ | Estimated future reward of an action in a state | Predicted link quality of a channel |
| [[bellman_equation]] | $r + \gamma \max_{a'}Q(s',a')$ — the regression target | Updating a channel estimate from a new pilot measurement |
| [[discount_factor]] $\gamma$ | Weights future reward against present | Forgetting factor in an EWMA over noisy RSSI |
| [[epsilon_greedy]] | Random action w.p. $\epsilon$, else greedy | Spectrum sensing (explore) vs. locking onto a known channel (exploit) |
| [[experience_replay]] | Random minibatch sampling from a transition buffer | Interleaving before FEC decoding — breaks temporal correlation |
| [[reward_shaping]] | Denser reward than the environment provides | Continuous error signal instead of a threshold alarm |
| Missing [[target_network]] | Same net supplies prediction *and* target | Tuning a filter against a reference that's also drifting |

### Hyperparameters

| Parameter | Value | Effect if raised |
|---|---|---|
| $\gamma$ (discount) | 0.95 | More far-sighted, but higher-variance targets |
| $\epsilon$ decay | 0.995 | Slower shift from exploring to exploiting |
| Replay capacity | 2000 | More diverse batches, more stale experiences |
| Batch size | 64 | Smoother gradients, slower per step |
| `train_frequency` | 5 | Less frequent training — cheaper, but slower learning |
| Learning rate | 0.001 | Faster movement, higher divergence risk |

### Key equations

$$Q(s, a) = r + \gamma \max_{a'} Q(s', a') \qquad \text{(Bellman)}$$

$$\mathcal{L}(\theta) = \big(Q_\theta(s,a) - y\big)^2 \qquad \text{(regression loss)}$$

$$H \approx \frac{1}{1-\gamma} \qquad \text{(effective planning horizon)}$$

### 🎓 What I take away

1. **RL is regression with self-generated, moving labels.** Once the Bellman target is seen as a
   label, the whole thing reduces to `model.fit()` — the novelty is in *where the label comes from*,
   not the training machinery.
2. **The engineering is in the data pipeline, not the model.** A 2-layer MLP is trivial. The replay
   buffer, exploration schedule, and reward design determine whether it learns at all — and Practice 1
   showed that widening the network barely registers by comparison.
3. **Exploration is a scheduled resource.** Open-loop schedules are predictable; closed-loop ones adapt
   but are only as good as the noisy signal driving them.
4. **Reward shaping changes the objective.** Anchoring the shaped reward to the environment's own
   termination thresholds is what makes it principled rather than arbitrary.
5. **This implementation is honest but not production-grade.** The gaps below are named, not hidden.

### ⚠️ Gaps to close next

| Gap | Consequence | Fix |
|---|---|---|
| No target network | Targets drift as weights update → unstable | Frozen copy of the network, synced every $N$ steps |
| `terminated` used as `done` in the Bellman branch | Truncation wrongly implies zero future value | Branch on `terminated` alone, not `terminated or truncated` |
| 10-episode training budget | Nowhere near solving CartPole; early stopping never fires | Several hundred episodes minimum |
| Single-seed comparisons | Practice 1 measures noise as much as architecture | Multiple seeds, report mean ± spread |

## 🧪 Sandbox

Space to experiment, roughly in order of value:

**1. Add a target network** — the single highest-impact fix. Keep a frozen copy of the model, compute
Bellman targets from *it*, and sync every $N$ steps:

```python
target_model = build_model(state_size, action_size)
target_model.set_weights(model.get_weights())

# in replay(): q_values_next = target_model.predict(next_states, verbose=0)
# every N steps:  target_model.set_weights(model.get_weights())
```

**2. Train for real** — raise `episodes` to 300–500 with the target network in place and watch whether
the early stopping check actually fires.

**3. Fix the terminal/truncation branch** — use `terminated` alone for the Bellman bootstrap so hitting
the time limit isn't misread as failure:

```python
next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated          # ends the episode
remember(state, action, reward, next_state, terminated)   # but bootstrap on terminated only
```

**4. Plot the learning curve** — collect episode scores and plot with a rolling mean. RL curves are
noisy enough that the raw trace is hard to read without smoothing.

**5. Make the adaptive epsilon robust** — trigger on a rolling mean of the last 10 episodes instead of
a single score, and lower the threshold below `max_timesteps` so it can actually fire.

**6. Sweep $\gamma$** — try 0.9, 0.95, 0.99 and check the effective-horizon rule $H \approx 1/(1-\gamma)$
against observed behaviour.

**7. Combine Practice 2 and 3** — adaptive epsilon plus the continuous reward in a single run.

**8. Try a harder environment** — `MountainCar-v0` or `Acrobot-v1` use the same interface but have far
sparser rewards, which makes exploration the dominant difficulty and shows why reward shaping matters.

In [ ]:
# 🧪 Sandbox — experiment freely